# Graph t-SNE

Compare graph-routing, node-feature temporal, and combined sample descriptors. Labels are loaded only after all embeddings are computed and are used solely for plot colors.

In [ ]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'cache.py').is_file():
    raise RuntimeError('Run this notebook from the repository root or its notebooks directory.')
sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

from cache import AttentionDataset, sha256
from descriptors import temporal_summary, token_routing_features
from features import load_node_features

ATTENTION_ROOT = Path('/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/model_traces/llama31_8b/test')
GRAPH_ROOT = Path('/share/home/tm902089733300000/a903202310/lys/data/RAGTruth/graphs/llama31_8b/relation_topk_channels/test')
NODE_FEATURE_MODE = 'attention'
RANDOM_STATE = 0
SAVE_DIR = None  # Set to a directory to save coordinates and the figure.

In [ ]:
graph_manifest = json.loads((GRAPH_ROOT / 'manifest.json').read_text(encoding='utf-8'))
attention_manifest = ATTENTION_ROOT / 'manifest.json'
attention_index = ATTENTION_ROOT / 'index.jsonl'
if (graph_manifest['input_manifest_sha256'] != sha256(attention_manifest)
        or graph_manifest['input_index_sha256'] != sha256(attention_index)):
    raise ValueError('Graph manifest does not match the attention split provenance hashes.')
graph_index = GRAPH_ROOT / 'index.jsonl'
if sha256(graph_index) != graph_manifest['index_sha256']:
    raise ValueError('Graph index_sha256 does not match manifest.json.')
graph_rows = {row['sample_id']: row for row in (json.loads(line) for line in graph_index.read_text(encoding='utf-8').splitlines() if line)}
dataset = AttentionDataset(ATTENTION_ROOT, device='cpu', verify_hashes=True)

def load_verified_graph(row):
    path = GRAPH_ROOT / row['path']
    if not path.is_file() or path.stat().st_size != row['bytes']:
        raise ValueError('Graph file byte count does not match index.jsonl.')
    if sha256(path) != row['sha256']:
        raise ValueError('Graph file SHA256 does not match index.jsonl.')
    return torch.load(path, map_location='cpu', weights_only=True)

sample_ids, routing_rows, node_rows = [], [], []
for sample in dataset:
    graph = load_verified_graph(graph_rows[sample.sample_id])
    routing = token_routing_features(graph, sample.num_channels)
    node_features = load_node_features(ATTENTION_ROOT, sample, NODE_FEATURE_MODE)
    sample_ids.append(sample.sample_id)
    routing_rows.append(temporal_summary(routing).numpy())
    node_rows.append(temporal_summary(node_features[sample.response_idx:]).numpy())

routing_matrix = np.stack(routing_rows)
node_matrix = np.stack(node_rows)
combined_matrix = np.concatenate((
    StandardScaler().fit_transform(routing_matrix) / np.sqrt(routing_matrix.shape[1]),
    StandardScaler().fit_transform(node_matrix) / np.sqrt(node_matrix.shape[1]),
), axis=1)

In [ ]:
def embed(matrix, pre_scaled=False):
    if len(matrix) < 2:
        raise ValueError('t-SNE needs at least two samples.')
    scaled = matrix if pre_scaled else StandardScaler().fit_transform(matrix)
    if scaled.shape[1] > 50:
        scaled = PCA(n_components=min(50, *scaled.shape), random_state=RANDOM_STATE).fit_transform(scaled)
    return TSNE(
        n_components=2, perplexity=min(30, len(scaled) - 1), init='pca',
        learning_rate='auto', max_iter=1000, random_state=RANDOM_STATE,
    ).fit_transform(scaled)

embeddings = {
    'routing': embed(routing_matrix),
    'node': embed(node_matrix),
    'combined': embed(combined_matrix, pre_scaled=True),
}

if SAVE_DIR is not None:
    SAVE_DIR.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(SAVE_DIR / 'graph_tsne_coordinates.npz', sample_id=np.asarray(sample_ids), **embeddings)

In [ ]:
labels = {
    row['sample_id']: int(bool(row['positive_runs']))
    for row in (json.loads(line) for line in (ATTENTION_ROOT / 'labels.jsonl').read_text(encoding='utf-8').splitlines() if line)
}
colors = np.asarray([labels[sample_id] for sample_id in sample_ids])

figure, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
for axis, (name, coordinates) in zip(axes, embeddings.items()):
    plot = axis.scatter(coordinates[:, 0], coordinates[:, 1], c=colors, cmap='coolwarm', alpha=0.8)
    axis.set(title=f'{name} t-SNE', xlabel='t-SNE 1', ylabel='t-SNE 2')
figure.colorbar(plot, ax=axes, label='positive response span')
if SAVE_DIR is None:
    plt.show()
else:
    figure.savefig(SAVE_DIR / 'graph_tsne.png', dpi=200, bbox_inches='tight')
    plt.close(figure)